# 9/4/2026

# code js giữ chân colab
 function ConnectButton(){
    console.log("Đang giữ kết nối...");
   document.querySelector("#top-toolbar > colab-connect-button").click()
}
setInterval(ConnectButton, 60000);



# import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Giải nén file images thành 2 file test và valid

In [ ]:
import os
import zipfile
import shutil
import cv2
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

def unzip_and_split_dataset(zip_path, base_extract_dir='/content/dataset'):
    """
    Hàm thực hiện giải nén file zip và chia ảnh vào 2 folder train/valid
    dựa vào từ khóa trong tên file.
    """
    print(f"Đang giải nén {zip_path}...")

    # 1. Giải nén file zip
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_extract_dir)
    print("Giải nén hoàn tất!")

    # 2. Tạo 2 thư mục đích để chứa ảnh
    train_dir = os.path.join(base_extract_dir, 'train_images')
    valid_dir = os.path.join(base_extract_dir, 'valid_images')

    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)

    count_train = 0
    count_valid = 0

    # 3. Duyệt qua các file vừa giải nén và phân loại
    print("Đang phân loại ảnh vào các thư mục...")
    for root, dirs, files in os.walk(base_extract_dir):
        # Bỏ qua nếu đang duyệt trúng chính 2 thư mục đích vừa tạo
        if root in [train_dir, valid_dir]:
            continue

        for file in files:
            # Chỉ xử lý các file ảnh
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
                file_path = os.path.join(root, file)

                # Kiểm tra tên file có chứa 'train' hay 'valid'
                if 'train' in file.lower():
                    shutil.move(file_path, os.path.join(train_dir, file))
                    count_train += 1
                elif 'valid' in file.lower():
                    shutil.move(file_path, os.path.join(valid_dir, file))
                    count_valid += 1

    print(f"Hoàn thành phân loại!")
    print(f" - Tổng số ảnh train: {count_train} -> Lưu tại: {train_dir}")
    print(f" - Tổng số ảnh valid: {count_valid} -> Lưu tại: {valid_dir}")

    return train_dir, valid_dir

# ----------------- CHẠY THỰC TẾ -----------------
zip_path = "/content/drive/MyDrive/images.zip"
train_folder, valid_folder = unzip_and_split_dataset(zip_path)

# Hàm load_images_from_folder của bạn ở đây...
# def load_images_from_folder(folder): ...

# 1. KHAI BÁO THƯ VIỆN & CẤU HÌNH THIẾT BỊ

In [ ]:

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import AutoTokenizer, ViTModel, GPT2LMHeadModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang sử dụng thiết bị: {device}")

#  2. ĐỌC DỮ LIỆU CSV 

In [ ]:
train_folder = '/content/dataset/train_images'
valid_folder = '/content/dataset/valid_images'
csv_path = "/content/drive/MyDrive/captions.csv"

print(f"Đang đọc file {csv_path}...")
df = pd.read_csv(csv_path)

# Tách file thành train và valid
train_df = df[df['ID'].str.contains('train', case=False)].reset_index(drop=True)
valid_df = df[df['ID'].str.contains('valid', case=False)].reset_index(drop=True)
print(f"-> Đã load {len(train_df)} dòng train, {len(valid_df)} dòng valid.")

# 3. KHỞI TẠO TOKENIZER & DATASET

In [ ]:
TOKENIZER_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME)
tokenizer.pad_token = tokenizer.eos_token

MOCK_CONCEPTS = ['opacity', 'effusion', 'pneumonia', 'fracture', 'normal', 'catheter', 'tube', 'lung', 'heart']
NUM_CONCEPTS = len(MOCK_CONCEPTS)

def extract_mock_concepts(caption):
    caption_lower = caption.lower()
    labels = [1.0 if concept in caption_lower else 0.0 for concept in MOCK_CONCEPTS]
    return torch.tensor(labels, dtype=torch.float)

class MultiTaskMedicalDataset(Dataset):
    def __init__(self, df, img_dir, tokenizer, transform=None, max_length=64):
        self.df = df
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'ID']
        caption = self.df.loc[idx, 'Caption']

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))

        if self.transform:
            image = self.transform(image)

        encoded_caption = self.tokenizer(
            caption,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        concept_labels = extract_mock_concepts(caption)

        return {
            "pixel_values": image,
            "input_ids": encoded_caption.input_ids.squeeze(),
            "attention_mask": encoded_caption.attention_mask.squeeze(),
            "concept_labels": concept_labels
        }

# Khởi tạo DataLoader
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = MultiTaskMedicalDataset(train_df, train_folder, tokenizer, basic_transform)
# Dùng num_workers=2 để load ảnh nhanh hơn
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)

# 4. KHỞI TẠO KIẾN TRÚC MÔ HÌNH (MODEL)

In [ ]:
class MedicalCaptioningMultiTaskModel(nn.Module):
    def __init__(self, num_concepts, vocab_size):
        super().__init__()
        self.encoder = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        vision_dim = self.encoder.config.hidden_size

        self.concept_head = nn.Sequential(
            nn.Linear(vision_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_concepts)
        )

        self.decoder = GPT2LMHeadModel.from_pretrained("distilgpt2")
        text_dim = self.decoder.config.n_embd

        self.bridge = nn.Linear(vision_dim, text_dim)

    def forward(self, pixel_values, input_ids=None, attention_mask=None):
        encoder_outputs = self.encoder(pixel_values=pixel_values)
        image_features = encoder_outputs.last_hidden_state[:, 0, :]

        concept_logits = self.concept_head(image_features)
        projected_image_features = self.bridge(image_features).unsqueeze(1)

        if input_ids is not None:
            inputs_embeds = self.decoder.transformer.wte(input_ids)
            combined_embeds = torch.cat((projected_image_features, inputs_embeds), dim=1)

            extended_attention_mask = torch.cat(
                (torch.ones(attention_mask.shape[0], 1).to(attention_mask.device), attention_mask), dim=1
            )

            labels = input_ids.clone()
            labels[attention_mask == 0] = -100

            image_label = torch.full((input_ids.shape[0], 1), -100, dtype=torch.long, device=input_ids.device)
            extended_labels = torch.cat((image_label, labels), dim=1)

            outputs = self.decoder(
                inputs_embeds=combined_embeds,
                attention_mask=extended_attention_mask,
                labels=extended_labels
            )
            caption_loss = outputs.loss
            return concept_logits, caption_loss

        return concept_logits, None

model = MedicalCaptioningMultiTaskModel(NUM_CONCEPTS, tokenizer.vocab_size).to(device)

#  5. VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP)

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
concept_loss_fn = nn.BCEWithLogitsLoss()

EPOCHS = 3
ALPHA = 1.0
BETA = 0.5

print("BẮT ĐẦU HUẤN LUYỆN...")

for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()

        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        concept_labels = batch["concept_labels"].to(device)

        concept_logits, caption_loss = model(pixel_values, input_ids, attention_mask)
        concept_loss = concept_loss_fn(concept_logits, concept_labels)

        loss = (ALPHA * caption_loss) + (BETA * concept_loss)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{EPOCHS}")
        loop.set_postfix(loss=loss.item(), cap_loss=caption_loss.item(), con_loss=concept_loss.item())

    avg_loss = total_train_loss / len(train_loader)
    print(f"\n=> Kết thúc Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

    save_dir = '/content/drive/MyDrive/Medical_Caption_Models'
    os.makedirs(save_dir, exist_ok=True)

    # Lưu trọng số của model
    checkpoint_path = os.path.join(save_dir, f'model_epoch_{epoch+1}.pt')
    torch.save(model.state_dict(), checkpoint_path)
    print(f" Đã lưu checkpoint an toàn tại: {checkpoint_path}")

print("Huấn luyện hoàn tất!")

# CODE Dự PHÒNG NẾU HẾT GPU FREE NÊN PHẢI CLONE SANG GMAIL KHÁC
## Tìm checkpoint, xong chạy tiếp trong 3 epoch

In [ ]:
# ==========================================
# 5. VÒNG LẶP HUẤN LUYỆN (TRAINING LOOP) - CÓ RESUME
# ==========================================
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)
concept_loss_fn = nn.BCEWithLogitsLoss()

TOTAL_EPOCHS = 3
ALPHA = 1.0
BETA = 0.5
save_dir = '/content/drive/MyDrive/Medical_Caption_Models'
os.makedirs(save_dir, exist_ok=True)

# --- THÊM MỚI: TỰ ĐỘNG TÌM VÀ NẠP CHECKPOINT ---
start_epoch = 0

# Tìm file có số epoch lớn nhất trong thư mục
existing_checkpoints = [f for f in os.listdir(save_dir) if f.startswith('model_epoch_') and f.endswith('.pt')]
if existing_checkpoints:
    # Lấy ra số epoch lớn nhất. Ví dụ: 'model_epoch_2.pt' -> lấy số 2
    epochs = [int(f.split('_')[-1].split('.')[0]) for f in existing_checkpoints]
    latest_epoch = max(epochs)

    # Nạp mô hình
    checkpoint_path = os.path.join(save_dir, f'model_epoch_{latest_epoch}.pt')
    print(f" Tìm thấy checkpoint! Đang nạp trí nhớ từ: {checkpoint_path}...")

    # map_location=device giúp nạp model an toàn dù bạn đang dùng CPU hay GPU
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))

    start_epoch = latest_epoch
    print(f" Đã nạp xong! Sẽ tiếp tục huấn luyện từ Epoch {start_epoch + 1}")
else:
    print("Mô hình mới tinh, sẽ huấn luyện từ Epoch 1.")
# -----------------------------------------------

print("BẮT ĐẦU HUẤN LUYỆN...")

# Sửa lại vòng lặp để bắt đầu từ start_epoch thay vì 0
for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    total_train_loss = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        optimizer.zero_grad()

        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        concept_labels = batch["concept_labels"].to(device)

        concept_logits, caption_loss = model(pixel_values, input_ids, attention_mask)
        concept_loss = concept_loss_fn(concept_logits, concept_labels)

        loss = (ALPHA * caption_loss) + (BETA * concept_loss)
        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()
        loop.set_description(f"Epoch {epoch+1}/{TOTAL_EPOCHS}")
        loop.set_postfix(loss=loss.item(), cap_loss=caption_loss.item(), con_loss=concept_loss.item())

    avg_loss = total_train_loss / len(train_loader)
    print(f"\n=> Kết thúc Epoch {epoch+1} | Average Loss: {avg_loss:.4f}")

    # Lưu trọng số của model
    checkpoint_path = os.path.join(save_dir, f'model_epoch_{epoch+1}.pt')
    torch.save(model.state_dict(), checkpoint_path)
    print(f" Đã lưu checkpoint tại: {checkpoint_path}")

print("Huấn luyện hoàn tất!")

# Khi đã huấn luyện xong epoch 3 ,ta lắp bộ não vào trong mô hình

# 6. Nạp lại thư viện cần thiết(có thể không cần)

In [ ]:
import os
import csv
import pandas as pd
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import AutoTokenizer, ViTModel, GPT2LMHeadModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {device}")

# 7.ĐỌC TẬP VALIDATION & TOKENIZER

In [ ]:
valid_folder = '/content/dataset/valid_images'
csv_path = "/content/drive/MyDrive/Colab Notebooks/captions.csv"

# Đọc file CSV
df = pd.read_csv(csv_path)
valid_df = df[df['ID'].str.contains('valid', case=False)].reset_index(drop=True)

# Khởi tạo Tokenizer và Transform
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token
NUM_CONCEPTS = 9 # Tương ứng với 9 mock concepts đã train

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 8.ĐỊNH NGHĨA LẠI KIẾN TRÚC MÔ HÌNH

In [ ]:
class MedicalCaptioningMultiTaskModel(nn.Module):
    def __init__(self, num_concepts, vocab_size):
        super().__init__()
        self.encoder = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        vision_dim = self.encoder.config.hidden_size

        self.concept_head = nn.Sequential(
            nn.Linear(vision_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_concepts)
        )

        self.decoder = GPT2LMHeadModel.from_pretrained("distilgpt2")
        text_dim = self.decoder.config.n_embd

        self.bridge = nn.Linear(vision_dim, text_dim)

    def forward(self, pixel_values, input_ids=None, attention_mask=None):
        pass # Rút gọn hàm forward vì inference không dùng tới bước tính Loss này


# 9. NẠP TRỌNG SỐ TỪ EPOCH 3

In [ ]:
# Khởi tạo bộ khung trống
model = MedicalCaptioningMultiTaskModel(NUM_CONCEPTS, tokenizer.vocab_size).to(device)

checkpoint_path = '/content/drive/MyDrive/Colab Notebooks/model_epoch_3.pt'
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
model.eval() # Chuyển sang chế độ dự đoán
print("Đã nạp thành công bộ não từ Epoch 3!")

# 10. HÀM SINH CÂU TỐI ƯU

In [ ]:
def generate_caption_optimized(model, tokenizer, pixel_values, max_length=60):
    with torch.no_grad():
        encoder_outputs = model.encoder(pixel_values=pixel_values)
        image_features = encoder_outputs.last_hidden_state[:, 0, :]
        projected_image = model.bridge(image_features).unsqueeze(1)

        generated_ids = model.decoder.generate(
            inputs_embeds=projected_image,
            max_new_tokens=max_length,
            num_beams=5,               # Dò 5 nhánh
            no_repeat_ngram_size=2,    # Phạt lỗi lặp từ
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )

        caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        return caption

# 11. DỰ ĐOÁN & XUẤT FILE

In [ ]:
print("Úm ba la...")
submission_data = []

with open('submission.csv', mode='w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(['ID', 'Caption'])
    for idx in tqdm(range(len(valid_df))):
        img_id = valid_df.loc[idx, 'ID']
        img_path = os.path.join(valid_folder, f"{img_id}.jpg")

        try:
            image = Image.open(img_path).convert('RGB')
            pixel_values = basic_transform(image).unsqueeze(0).to(device)
            pred_caption = generate_caption_optimized(model, tokenizer, pixel_values)
        except Exception as e:
            pred_caption = ""

        writer.writerow([img_id, pred_caption])
        submission_data.append({"ID": img_id, "Pred": pred_caption, "True": valid_df.loc[idx, 'Caption']})

print("\n=== KẾT QUẢ VÍ DỤ ===")
for i in range(3):
    print(f"Ảnh {i+1}:")
    print(f"  Thực tế : {submission_data[i]['True']}")
    print(f"  Mô hình : {submission_data[i]['Pred']}\n")

# 12. Tự chấm điểm(9/4/2026)

In [ ]:

!pip install -q evaluate rouge_score bert_score

import evaluate
import numpy as np

# Load các hàm tính điểm
rouge = evaluate.load('rouge')
bertscore = evaluate.load('bertscore')

# Lấy toàn bộ kết quả để chấm
predictions = [item["Pred"] for item in submission_data]
references = [item["True"] for item in submission_data]

# 1. TÍNH ĐIỂM ROUGE-1 (F-measure)
# Ban tổ chức kiểm tra độ trùng lặp từng từ (unigrams)
print("\n1. Đang tính điểm ROUGE-1...")
rouge_results = rouge.compute(predictions=predictions, references=references)
rouge1_score = rouge_results['rouge1']
print(f" ROUGE-1 (F-measure): {rouge1_score:.4f} (Càng gần 1 càng tốt)")

# ... (Phần 1 tính ROUGE giữ nguyên) ...

# 2. TÍNH ĐIỂM BERTSCORE (Recall)
print("\n2. Đang tính điểm BERTScore (Đã đổi sang DistilBERT để fix lỗi Overflow)...")
bert_results = bertscore.compute(
    predictions=predictions,
    references=references,
    lang="en",
    model_type="distilbert-base-uncased" # Đổi giám khảo ở dòng này
)

# BTC yêu cầu lấy điểm Recall
avg_bert_recall = np.mean(bert_results['recall'])
print(f" BERTScore (Recall): {avg_bert_recall:.4f} (Càng gần 1 càng tốt)")

print("\n===============================")
print(f" ĐIỂM TRUNG BÌNH (RELEVANCE): {(rouge1_score + avg_bert_recall) / 2:.4f}")
print("===============================")

# 13 Trực quan hóa

In [ ]:
import matplotlib.pyplot as plt
import textwrap

print("ĐANG TRÍCH XUẤT 3 VÍ DỤ MINH HỌA...")

# Lấy 3 mẫu đầu tiên từ list submission_data mà bạn đã sinh ra ở Cell trước
samples = submission_data[:3]

fig, axes = plt.subplots(3, 1, figsize=(18, 18))

for i, sample in enumerate(samples):
    img_id = sample['ID']
    true_cap = sample['True']
    pred_cap = sample['Pred']

    # Load ảnh
    img_path = os.path.join(valid_folder, f"{img_id}.jpg")
    try:
        img = Image.open(img_path)
    except:
        img = Image.new('RGB', (112, 112), color='gray') # Ảnh xám nếu lỗi

    # Bẻ dòng text cho đỡ bị tràn màn hình (mỗi dòng 80 ký tự)
    wrapped_true = textwrap.fill(f"THỰC TẾ: {true_cap}", width=160)
    wrapped_pred = textwrap.fill(f"MÔ HÌNH: {pred_cap}", width=160)

    # Hiển thị
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(f"ID: {img_id}", fontsize=12, fontweight='bold')

    # In text ngay dưới ảnh
    axes[i].text(0, img.size[1] + 20, wrapped_true, fontsize=11, color='green', verticalalignment='top')
    axes[i].text(0, img.size[1] + 60, wrapped_pred, fontsize=11, color='blue', verticalalignment='top')

plt.tight_layout()
plt.show()

# 10/4/2026

# vì bản 9/4 khi chấm điểm, file scoring result trống trơn ==> NULL 
### chạy rất mượt qua 5 vòng (BERTScore, AlignScore, ROUGE, Similarity, BLEURT), nhưng đến dòng: Compute MedCAT + Computing MEDCATS thì nó đột ngột dừng lại và không in ra thêm gì nữa.
### Cùng lúc đó, file metadata trả về kết quả:
### elapsedTime: null
### exitCode: null
# Nguyên nhân: Quá trình chấm điểm đã bị hệ thống "giết" giữa chừng ngay tại bước chạy MedCAT. Vì tiến trình bị ngắt đột ngột, nó không kịp chạy đến dòng code cuối cùng để ghi file kết quả tổng =>file trả về bị trống.
# Lý do bị Kill: MedCAT là một mô hình cực kỳ ngốn RAM (vì nó phải load toàn bộ bộ từ điển y khoa khổng lồ UMLS vào bộ nhớ). Việc quét MedCAT cho 19k câu caption thường gây ra hiện tượng Tràn bộ nhớ / Timeout trên máy chủ hệ thống

## ===>Đây là lỗi từ phía BTC chứ không phải do file submission.csv bị sai (vì hệ thống đã báo Submission format check passed ở ngay đầu).

### Cách xử lý: Trong file README có ghi rõ: "For questions regarding the evaluation scripts please use the challenge website forum or contact tabea.pakull@uk-essen.de". Nên gửi một email/post lên forum báo cho BTC rằng: "Hệ thống chấm điểm đang bị crash (OOM/Timeout) ở bước Computing MEDCATS đối với file submission đầy đủ 19.239 dòng". Họ sẽ phải tăng RAM cho Docker server của họ lên.

# Dù bị crash ở phút chót, nhưng may mắn là hệ thống đã kịp in ra 5 điểm số đầu tiên lên log. Hãy cùng nhìn vào thành quả của 3 Epochs:

### ROUGE: 0.4713 (47.1%) -> Điểm này đo số lượng từ vựng trùng khớp. 47% là con số rất cao cho bài toán này.

### BERTScore: 0.6618 (66.1%) -> Giám khảo DeBERTa của BTC chấm câu của bạn giống 66% ý nghĩa so với bác sĩ thực tế.

### Similarity (MedImageInsights): 0.7577 (75.7%) -> Đây là điểm rất quan trọng! BTC dùng mô hình nhúng (embedding) chuyên ngành y khoa để so khớp. Điểm 75.7% cho thấy AI  đã miêu tả rất sát với bức ảnh gốc.

### BLEURT: 0.3722 -> Đây là chỉ số đánh giá độ tự nhiên của văn phong (giống người viết). Điểm BLEURT thường khá khắt khe, đạt trên 0.3 cho Baseline là một tín hiệu mừng.

# Lưu ý duy nhất: Điểm AlignScore: 0.2026 (Đo lường độ chính xác thông tin y khoa - Factuality) đang bị thấp. Điều này hoàn toàn dễ hiểu vì mô hình của chúng ta mới chỉ đang dùng 9 từ khóa MOCK_CONCEPTS giả lập.

# Ngày 1 đợi có chứng chỉ + file umls_self_train_model...zip của BTC

### thay 9 từ MOCK_CONCEPTS thành các mã UMLS thực tế từ tập Train.

# ScípaCy
### SciSpaCy cần các thư viện chuyên dụng về ngôn ngữ học và một bộ từ điển UMLS rút gọn (khoảng 1GB) để hoạt động. Bạn chạy Cell này để cài đặt (mất khoảng 2-3 phút):

In [ ]:
# Cài đặt phiên bản spaCy tương thích và scispacy
!pip install -q spacy==3.4.4
!pip install -q scispacy==0.5.1
!pip install -q nmslib # Thư viện hỗ trợ tìm kiếm CUI nhanh

# Tải mô hình ngôn ngữ y khoa của SciSpaCy
!pip install -q https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.1/en_core_sci_sm-0.5.1.tar.gz

print("Cài đặt hoàn tất!")

# Quét tập Train và Tạo bộ nhãn (UMLS CUIs)

# Quét tập Train và Tạo bộ nhãn (UMLS CUIs)
### đọc file captions.csv, lôi từng câu ra cho SciSpaCy đọc để bắt bệnh, trích xuất mã CUI (VD: C0032285), và lưu thành file captions_with_cuis.csv lên Google Drive.

In [ ]:
import pandas as pd
import spacy
import scispacy
from scispacy.linking import EntityLinker
from collections import Counter
from tqdm import tqdm
import os

print("Đang khởi tạo AI Y khoa SciSpaCy (Lần đầu chạy sẽ tải bộ UMLS hơi lâu chút)...")
# Load mô hình lõi
nlp = spacy.load("en_core_sci_sm")

# Thêm "bộ não" tra cứu UMLS vào pipeline
nlp.add_pipe("scispacy_linker", config={"resolve_abbreviations": True, "linker_name": "umls"})
linker = nlp.get_pipe("scispacy_linker")

# Đọc file gốc
csv_path = "/content/drive/MyDrive/captions.csv"
df = pd.read_csv(csv_path)

# Hàm rút trích CUI từ 1 câu
def extract_cuis(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    doc = nlp(text)
    cuis = []
    for ent in doc.ents:
        if ent._.kb_ents:
            # Lấy CUI có độ tin cậy cao nhất (top 1) cho thực thể này
            best_cui = ent._.kb_ents[0][0]
            cuis.append(best_cui)
    # Loại bỏ CUI trùng lặp trong cùng 1 câu và ghép thành chuỗi cách nhau bởi dấu chấm phẩy
    return ";".join(list(set(cuis)))

print(f"Đang quét {len(df)} câu caption để tìm mã bệnh lý...")
# Để tránh Colab quá tải, mình lưu kết quả vào một list trước
all_cuis = []

# Mẹo: Quá trình này có thể tốn 30p - 1 tiếng. Nếu bạn muốn test code trước, 
# hãy thay df.iterrows() thành df.head(1000).iterrows() để chạy thử 1000 dòng.
for idx, row in tqdm(df.iterrows(), total=len(df)):
    cuis_str = extract_cuis(row['Caption'])
    all_cuis.append(cuis_str)

# Gắn cột mới vào DataFrame
df['CUIs'] = all_cuis

# LƯU THÀNH FILE MỚI ĐỂ DÙNG MÃI MÃI
new_csv_path = "/content/drive/MyDrive/captions_with_cuis.csv"
df.to_csv(new_csv_path, index=False)
print(f"Đã quét xong và lưu data mới tại: {new_csv_path}")

# Thay MOCK bằng REAL
## Sau khi có file mới, chúng ta sẽ lọc ra Top 200 mã CUI xuất hiện nhiều nhất trong tập Train để làm "Từ vựng bệnh lý" (Classes) cho mô hình dự đoán. Các mã hiếm quá (chỉ xuất hiện 1-2 lần) sẽ bị bỏ qua để tránh nhiễu.

In [ ]:
import ast
import torch

print("ĐANG XÂY DỰNG TỪ ĐIỂN UMLS...")
# 1. Đọc file vừa tạo
df_new = pd.read_csv("/content/drive/MyDrive/captions_with_cuis.csv")
df_new['CUIs'] = df_new['CUIs'].fillna("")

# 2. Đếm tần suất các CUI trong tập TRAIN để tìm top 200
train_cuis_series = df_new[df_new['ID'].str.contains('train', case=False)]['CUIs']
all_train_cuis = []
for cuis_str in train_cuis_series:
    if cuis_str:
        all_train_cuis.extend(cuis_str.split(";"))

cui_counts = Counter(all_train_cuis)
TOP_K = 200 # Số lượng bệnh lý muốn model học (bạn có thể tăng/giảm tùy ý)
REAL_CONCEPTS = [cui for cui, count in cui_counts.most_common(TOP_K)]
NUM_CONCEPTS = len(REAL_CONCEPTS)

print(f" Đã chốt danh sách {NUM_CONCEPTS} bệnh lý phổ biến nhất (Ví dụ: {REAL_CONCEPTS[:5]})")

# 3. Hàm tạo Multi-hot vector dựa trên đồ thật
def extract_real_concepts(cui_string):
    if not cui_string:
        return torch.zeros(NUM_CONCEPTS, dtype=torch.float)
        
    cui_list = cui_string.split(";")
    labels = [1.0 if concept in cui_list else 0.0 for concept in REAL_CONCEPTS]
    return torch.tensor(labels, dtype=torch.float)

# 4. Cập nhật class Dataset
class MultiTaskMedicalDataset(Dataset):
    def __init__(self, df, img_dir, tokenizer, transform=None, max_length=64):
        self.df = df
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'ID']
        caption = self.df.loc[idx, 'Caption']
        cui_string = self.df.loc[idx, 'CUIs'] # Lấy nhãn thật từ CSV

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224)) 

        if self.transform:
            image = self.transform(image)

        encoded_caption = self.tokenizer(
            caption,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        # Dùng nhãn thật thay vì mock
        concept_labels = extract_real_concepts(cui_string)

        return {
            "pixel_values": image,
            "input_ids": encoded_caption.input_ids.squeeze(),
            "attention_mask": encoded_caption.attention_mask.squeeze(),
            "concept_labels": concept_labels
        }

# (Sau đó bạn khởi tạo lại train_loader, valid_loader và khai báo Model với NUM_CONCEPTS mới như cũ)

## Train lại .....

# 11/4/2026

# HÀNG REAL VỀ

In [ ]:

!pip install -q transformers evaluate rouge_score bert_score

import os
import csv
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
from transformers import AutoTokenizer, ViTModel, GPT2LMHeadModel
import evaluate
import numpy as np

# Bật GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f" Đang chạy trên: {device}")


In [ ]:
import os
import zipfile
import shutil
import cv2
from google.colab import drive
drive.mount('/content/drive')
def unzip_and_split_dataset(zip_path, base_extract_dir='/content/dataset'):
    print(f"Đang giải nén {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(base_extract_dir)
    train_dir = os.path.join(base_extract_dir, 'train_images')
    valid_dir = os.path.join(base_extract_dir, 'valid_images')
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(valid_dir, exist_ok=True)
    count_train = 0
    count_valid = 0
    for root, dirs, files in os.walk(base_extract_dir):
        if root in [train_dir, valid_dir]:
            continue
        for file in files:
            # Chỉ xử lý các file ảnh
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
                file_path = os.path.join(root, file)
                if 'train' in file.lower():
                    shutil.move(file_path, os.path.join(train_dir, file))
                    count_train += 1
                elif 'valid' in file.lower():
                    shutil.move(file_path, os.path.join(valid_dir, file))
                    count_valid += 1
    print(f" - Tổng số ảnh train: {count_train} -> Lưu tại: {train_dir}")
    print(f" - Tổng số ảnh valid: {count_valid} -> Lưu tại: {valid_dir}")
    return train_dir, valid_dir
zip_path = "/content/drive/MyDrive/images.zip"
train_folder, valid_folder = unzip_and_split_dataset(zip_path)


In [ ]:
ZIP_PATH = '/content/drive/MyDrive/download.zip'
EXTRACT_DIR = '/content/download_sciebo'

print(f" Bắt đầu giải nén file {ZIP_PATH}...")
os.makedirs(EXTRACT_DIR, exist_ok=True)

try:
    # 1. Giải nén file zip gốc
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    for root, dirs, files in os.walk(EXTRACT_DIR):
        for file in files:
            if file.endswith('.zip'):
                nested_zip_path = os.path.join(root, file)
                print(f" Phát hiện file zip lồng nhau: {file}. Đang bung file...")
                with zipfile.ZipFile(nested_zip_path, 'r') as zip_ref:
                    # Giải nén ngay tại thư mục chứa file zip đó
                    zip_ref.extractall(root)

    print("\n GIẢI NÉN HOÀN TẤT!")
    print("\n CÂY THƯ MỤC CÁC FILE CSV VỪA TẠO:")
    for root, dirs, files in os.walk(EXTRACT_DIR):
        level = root.replace(EXTRACT_DIR, '').count(os.sep)
        indent = ' ' * 4 * (level)
        folder_name = os.path.basename(root)
        if folder_name:
            print(f"{indent} {folder_name}/")

        subindent = ' ' * 4 * (level + 1)
        for f in files:
            # Chỉ in ra các file csv (bỏ qua file zip gốc đã giải nén)
            if f.endswith('.csv'):
                file_path = os.path.join(root, f)
                print(f"{subindent}📄 {f} ---> (Copy path: {file_path})")

except FileNotFoundError:
    print(f" LỖI: Không tìm thấy file tại đường dẫn '{ZIP_PATH}'. Bạn hãy kiểm tra lại xem file đã tải lên Drive chưa nhé!")
except Exception as e:
    print(f" LỖI KHÔNG XÁC ĐỊNH: {e}")

In [ ]:
CAPTIONS_CSV = '/content/drive/MyDrive/captions.csv'
CUI_VOCAB_PATH = '/content/download_sciebo/cui_to_name_synth_2026.csv'
TRAIN_CONCEPTS_PATH = '/content/download_sciebo/manuel_concepts/train_concepts_manual.csv'
VALID_CONCEPTS_PATH = '/content/download_sciebo/manuel_concepts/valid_concepts_manual.csv'
TRAIN_IMG_DIR = '/content/dataset/train_images'
VALID_IMG_DIR = '/content/dataset/valid_images'

cui_vocab_df = pd.read_csv(CUI_VOCAB_PATH)
REAL_CONCEPTS = cui_vocab_df['CUI'].tolist()
NUM_CONCEPTS = len(REAL_CONCEPTS)

concept_dict = {}

def load_and_fix_concepts(csv_path, split_name):
    df = pd.read_csv(csv_path)
    for idx, row in df.iterrows():
        img_id = row['ID']
        if 'synth' not in img_id:
            img_id = img_id.replace(f'_{split_name}_', f'_synth_{split_name}_')
        cuis = str(row['CUIs']).split(';') if pd.notna(row['CUIs']) else []
        concept_dict[img_id] = cuis

load_and_fix_concepts(TRAIN_CONCEPTS_PATH, 'train')
load_and_fix_concepts(VALID_CONCEPTS_PATH, 'valid')

df_all = pd.read_csv(CAPTIONS_CSV)
synth_df = df_all[df_all['ID'].str.contains('synth', case=False)].reset_index(drop=True)

train_df = synth_df[synth_df['ID'].str.contains('train', case=False)].reset_index(drop=True)
valid_df = synth_df[synth_df['ID'].str.contains('valid', case=False)].reset_index(drop=True)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")
tokenizer.pad_token = tokenizer.eos_token

basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
class MultiTaskMedicalDataset_Pro(Dataset):
    def __init__(self, df, img_dir, tokenizer, concept_dict, transform=None, max_length=64):
        self.df = df
        self.img_dir = img_dir
        self.tokenizer = tokenizer
        self.concept_dict = concept_dict
        self.transform = transform
        self.max_length = max_length

    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        img_id = self.df.loc[idx, 'ID']
        caption = self.df.loc[idx, 'Caption']

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224))
        if self.transform:
            image = self.transform(image)

        encoded_caption = self.tokenizer(
            caption, padding='max_length', truncation=True,
            max_length=self.max_length, return_tensors="pt"
        )

        true_cuis = self.concept_dict.get(img_id, [])
        labels = [1.0 if concept in true_cuis else 0.0 for concept in REAL_CONCEPTS]
        concept_labels = torch.tensor(labels, dtype=torch.float)

        return {
            "pixel_values": image,
            "input_ids": encoded_caption.input_ids.squeeze(),
            "attention_mask": encoded_caption.attention_mask.squeeze(),
            "concept_labels": concept_labels
        }

train_dataset = MultiTaskMedicalDataset_Pro(train_df, TRAIN_IMG_DIR, tokenizer, concept_dict, basic_transform)
valid_dataset = MultiTaskMedicalDataset_Pro(valid_df, VALID_IMG_DIR, tokenizer, concept_dict, basic_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=2)

In [ ]:
class MedicalCaptioningMultiTaskModel(nn.Module):
    def __init__(self, num_concepts, vocab_size):
        super().__init__()
        self.encoder = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
        vision_dim = self.encoder.config.hidden_size

        self.concept_head = nn.Sequential(
            nn.Linear(vision_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_concepts)
        )

        self.decoder = GPT2LMHeadModel.from_pretrained("distilgpt2")
        text_dim = self.decoder.config.n_embd
        self.bridge = nn.Linear(vision_dim, text_dim)

    def forward(self, pixel_values, input_ids=None, attention_mask=None):
        encoder_outputs = self.encoder(pixel_values=pixel_values)
        image_features = encoder_outputs.last_hidden_state[:, 0, :]

        concept_logits = self.concept_head(image_features)

        projected_image = self.bridge(image_features).unsqueeze(1)

        text_loss = None
        if input_ids is not None:
            input_embeds = self.decoder.transformer.wte(input_ids)
            combined_embeds = torch.cat((projected_image, input_embeds[:, :-1, :]), dim=1)
            decoder_outputs = self.decoder(inputs_embeds=combined_embeds, attention_mask=attention_mask, labels=input_ids)
            text_loss = decoder_outputs.loss

        return text_loss, concept_logits

model = MedicalCaptioningMultiTaskModel(NUM_CONCEPTS, tokenizer.vocab_size).to(device)

old_weights_path = '/content/drive/MyDrive/model_synth_real_cui_epoch_3.pt'
if os.path.exists(old_weights_path):
    state_dict = torch.load(old_weights_path, map_location=device)
    filtered_dict = {k: v for k, v in state_dict.items() if 'concept_head' not in k}
    model.load_state_dict(filtered_dict, strict=False)
    print("Loaded Vision and Text weights from Epoch 3. Concept Head initialized randomly.")

# checkpoint csv

In [ ]:
import os
import csv
import torch
import pandas as pd
import glob
from tqdm import tqdm
from PIL import Image

IMG_DIR = '/content/dataset/valid_images' 
SAVE_DIR = '/content/drive/MyDrive'
MAIN_FILE = os.path.join(SAVE_DIR, 'submission_main.csv')
SAVE_STEP = 3000

# ==========================================
# 1. NẠP TRỌNG SỐ TỪ LẦN TRAIN XỊN NHẤT
# ==========================================
best_model_path = '/content/drive/MyDrive/model_synth_real_cui_epoch_3.pt'
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print(" Đã nạp thành công bộ não xịn nhất!")
else:
    print(" Chưa tìm thấy file.")
model.eval()

# ==========================================
# 2. HÀM SINH CÂU
# ==========================================
def generate_caption_optimized(model, tokenizer, pixel_values, max_length=60):
    with torch.no_grad():
        encoder_outputs = model.encoder(pixel_values=pixel_values)
        image_features = encoder_outputs.last_hidden_state[:, 0, :]
        projected_image = model.bridge(image_features).unsqueeze(1)
        generated_ids = model.decoder.generate(
            inputs_embeds=projected_image,
            max_new_tokens=max_length,
            num_beams=5,               
            no_repeat_ngram_size=2,    
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
        )
        caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        return caption

# ==========================================
# 3. QUÉT THÔNG MINH TOÀN BỘ FILE CŨ
# ==========================================
print(" Đang quét TOÀN BỘ các file Checkpoint để gom tiến độ...")
processed_ids = set()
all_results = []

# Tìm TẤT CẢ các file csv bắt đầu bằng chữ submission trong Drive
csv_files = glob.glob(os.path.join(SAVE_DIR, 'submission*.csv'))

for file in csv_files:
    try:
        df_temp = pd.read_csv(file)
        for _, row in df_temp.iterrows():
            if row['ID'] not in processed_ids:
                processed_ids.add(row['ID'])
                all_results.append([row['ID'], row['Caption']])
    except Exception as e:
        pass

print(f"Đã gom thành công {len(processed_ids)} câu từ các file Checkpoint!")

if len(all_results) > 0:
    temp_df = pd.DataFrame(all_results, columns=['ID', 'Caption'])
    temp_df.to_csv(MAIN_FILE, index=False, quoting=csv.QUOTE_ALL)

remaining_df = valid_df[~valid_df['ID'].isin(processed_ids)].reset_index(drop=True)

if len(remaining_df) == 0:
    print(" Mọi thứ đã hoàn thiện 100%, không cần chạy thêm!")
else:
    print(f" Bắt đầu chạy {len(remaining_df)} câu còn lại...")
    total_done = len(processed_ids)
    
    with open(MAIN_FILE, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        
        for idx in tqdm(range(len(remaining_df))):
            img_id = remaining_df.loc[idx, 'ID']
            img_path = os.path.join(IMG_DIR, f"{img_id}.jpg")
            
            try:
                image = Image.open(img_path).convert('RGB')
                pixel_values = basic_transform(image).unsqueeze(0).to(device) 
                pred_caption = generate_caption_optimized(model, tokenizer, pixel_values)
            except Exception as e:
                pred_caption = ""
                
            writer.writerow([img_id, pred_caption])
            f.flush() 
            
            all_results.append([img_id, pred_caption])
            total_done += 1
            
            if total_done % SAVE_STEP == 0:
                milestone_num = total_done // SAVE_STEP
                checkpoint_name = f'submission_m{milestone_num}_({total_done}_cau).csv'
                checkpoint_path = os.path.join(SAVE_DIR, checkpoint_name)
                
                pd.DataFrame(all_results, columns=['ID', 'Caption']).to_csv(checkpoint_path, index=False, quoting=csv.QUOTE_ALL)
                print(f"\n Đã lưu mốc {total_done} câu vào: {checkpoint_name}")

    final_path = os.path.join(SAVE_DIR, f'submission_FINAL_({total_done}_cau).csv')
    pd.DataFrame(all_results, columns=['ID', 'Caption']).to_csv(final_path, index=False, quoting=csv.QUOTE_ALL)
    print(f"\n XONG TRỌN VẸN 100%! File hoàn chỉnh nằm tại: {final_path}")

# điểm đánh giá


### ovr 0.3764
### score 0.4452
### score_secondary 0.3076
### bert 0.5196
### rouge 0.2876
### similarity 0.6569
### bleurt 0.3169
### medcat 0.2206
### align 0.3947

# NHẬN XÉT ĐÁNH GIÁ
1. ĐIỂM SÁNG : Tính chính xác Y khoa đã tăng vọt
Mục tiêu của đợt "đập đi xây lại" vừa rồi là dùng bộ mã CUI xịn của Ban Tổ Chức (BTC) để ép mô hình nói đúng bệnh lý. Và kết quả đã chứng minh chiến thuật của chúng ta hoàn toàn chính xác:

AlignScore (0.3947): Lần trước điểm này chỉ lẹt đẹt ở mức 0.20. Việc nó tăng gần gấp đôi lên ~0.40 chứng tỏ mô hình không còn "chém gió" bậy bạ nữa, các thông tin y khoa nó sinh ra đã bám rất sát với sự thật của bức ảnh.

MedCAT (0.2206): Lần trước hệ thống bị crash nên điểm bằng 0. Bây giờ đã đạt 22%. Với một bài toán dự đoán hàng ngàn mã bệnh lý (Concepts) phức tạp, việc mô hình nhỏ gọn đoán trúng được 22% mã CUI chuẩn của bác sĩ là một sự khởi đầu vô cùng vững chắc!

2. SỰ ĐÁNH ĐỔI:Relevance bị giảm sút 
Chắc bạn cũng nhận ra các điểm ROUGE (0.2876), BERT (0.5196), Similarity (0.6569) đều bị giảm so với cái Baseline 9 từ khóa giả lập ban đầu. Tại sao lại như vậy?

Lý do: Biến CONCEPT_WEIGHT = 2.0 ở trong vòng lặp Train. Ta đã ép mô hình phải dồn gấp đôi sự chú ý (trọng số) vào việc "Đoán đúng bệnh"---> việc nó lơ là việc "Học cách hành văn cho hay".

Nó giống như một sinh viên y khoa đang cố gắng nhồi nhét quá nhiều từ vựng chuyên ngành vào đầu, nên khi nói chuyện bị lắp bắp, câu chữ khô khan, không được trôi chảy tự nhiên như người bình thường. Điểm ROUGE giảm là vì nó dùng đúng từ khóa y khoa nhưng lại ghép câu không giống hệt cách bác sĩ viết.

# Hướng phát triển
## Chiến thuật 1: Giảm áp lực cho nhánh phân loại bệnh(Tăng ROUGE)
#### Lần tới khi Train, thử sửa CONCEPT_WEIGHT = 1.0 (thay vì 2.0). Điều này giúp mô hình chia đều 50% trí não cho việc học văn phong và 50% cho việc đoán bệnh. Điểm ROUGE sẽ tăng mạnh trở lại.

## Chiến thuật 2: Tăng số Epochs 
#### Lúc trước chúng ta chỉ có 9 nhãn bệnh lý, học 3 Epochs là thuộc. Bây giờ số lượng nhãn lên tới hàng ngàn, 3 Epochs là quá ít để mạng ViT + GPT2 có thể tiêu hóa hết. Nếu có thời gian cắm máy, bạn hãy set EPOCHS = 5 hoặc 10. Mô hình chắc chắn sẽ khôn lên rất nhiều.

## Chiến thuật 3: Hack thủ thuật sinh câu (Không cần Train lại)
#### Bạn có thể tinh chỉnh hàm generate_caption_optimized một chút và sinh ra file CSV mới ngay trên trọng số hiện tại:
#### Thử giảm num_beams=3 (cho câu sinh ra đa dạng hơn, ít bị rập khuôn).
#### Thử thêm temperature=0.8 để câu văn mềm mại hơn.

# điểm chiến thuật 3: ScoresActions471submission_FINAL_19239_cau_1.zip2026-04-11 15:04Finished
### ovr 0.3716
### score 0.4455
### score_secondary 0.2977
### bert 0.5188
### rouge 0.2876
### similarity 0.6586
### bleurt 0.3168
### medcat 0.2180
### align 0.3774

Phân tích kết quả thí nghiệm:
Điểm ROUGE (0.2876): Hoàn toàn không nhúc nhích. Nghĩa là việc ép mô hình dùng từ ngữ phong phú hơn không giúp nó khớp với các n-gram (cụm từ) của đáp án gốc.

Điểm MedCAT (0.2206 ⬇️ 0.2180) & Align (0.3947 ⬇️ 0.3774): Cả 2 điểm về tính chính xác y khoa đều bị giảm.
=> Kết luận rút ra: Trong lĩnh vực y khoa, sự "sáng tạo" (temperature) là một con dao hai lưỡi. Khi chúng ta thả lỏng cho GPT-2 tự do chọn từ, nó không chọn được từ chuyên ngành hay hơn, mà nó lại chọn những từ chung chung, làm câu văn bớt chuẩn xác đi (giảm MedCAT) mà văn phong cũng không khá hơn (ROUGE đứng im). Bài học đắt giá là với y khoa, chế độ Beam Search thuần túy (như ban đầu) vẫn là chân ái!

 # 12/4/2026

In [ ]:
import os
import torch
from torch.optim import AdamW
from tqdm import tqdm
import torch.nn as nn

# ==========================================
# CẤU HÌNH HUẤN LUYỆN NÂNG CAO (Chạy nốt Epoch 6)
# ==========================================
EPOCHS = 6            # Tổng số Epoch muốn đạt tới
START_EPOCH = 4       # Bắt đầu chạy từ Epoch 6 (vì đã có file của Epoch 4)
SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks'

optimizer = AdamW(model.parameters(), lr=5e-5)
text_criterion = nn.CrossEntropyLoss()
concept_criterion = nn.BCEWithLogitsLoss()

# Giữ nguyên trọng số 1.0 để model tập trung trau chuốt văn phong
CONCEPT_WEIGHT = 1.0 

# Nạp lại trí nhớ của Epoch 5 vừa bị ngắt
checkpoint_path = os.path.join(SAVE_DIR, 'model_synth_real_cui_epoch_5.pt')

if os.path.exists(checkpoint_path):
    print(f"🧠 Đang nạp lại trí nhớ từ {checkpoint_path}...")
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("✅ Nạp thành công! Sẵn sàng học nốt chặng cuối.")
else:
    print(f"❌ Không tìm thấy file tại: {checkpoint_path}. Bạn kiểm tra lại Drive xem file epoch_5.pt đã được lưu chưa nhé!")

# ==========================================
# VÒNG LẶP HUẤN LUYỆN
# ==========================================
for epoch in range(START_EPOCH, EPOCHS):
    model.train()
    total_loss = 0
    total_text_loss = 0
    total_concept_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for batch in progress_bar:
        optimizer.zero_grad()
        
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        concept_labels = batch["concept_labels"].to(device)
        
        text_loss, concept_logits = model(pixel_values, input_ids, attention_mask)
        concept_loss = concept_criterion(concept_logits, concept_labels)
        
        # Gộp Loss với trọng số mới (1.0)
        loss = text_loss + (CONCEPT_WEIGHT * concept_loss)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_text_loss += text_loss.item()
        total_concept_loss += concept_loss.item()
        
        progress_bar.set_postfix({
            'Loss': f"{loss.item():.4f}", 
            'Text': f"{text_loss.item():.4f}", 
            'CUI': f"{concept_loss.item():.4f}"
        })
        
    print(f"\n--- Hết Epoch {epoch+1} ---")
    print(f"Avg Loss: {total_loss/len(train_loader):.4f} | Text Loss: {total_text_loss/len(train_loader):.4f} | CUI Loss: {total_concept_loss/len(train_loader):.4f}")
    
    save_path = os.path.join(SAVE_DIR, f'model_synth_real_cui_epoch_{epoch+1}.pt')
    torch.save(model.state_dict(), save_path)
    print(f"💾 Đã lưu model tại: {save_path}\n")

In [ ]:
import os
import csv
import torch
import pandas as pd
import glob
from tqdm import tqdm
from PIL import Image

# ĐƯỜNG DẪN GỐC
IMG_DIR = '/content/dataset/valid_images' 
SAVE_DIR = '/content/drive/MyDrive/Colab Notebooks'

# Tên file mới hoàn toàn để không bị lộn với Epoch 3
MAIN_FILE = os.path.join(SAVE_DIR, 'submission_epoch6_main.csv')
SAVE_STEP = 3000

# ==========================================
# 1. NẠP TRỌNG SỐ TỪ EPOCH 6 XỊN NHẤT
# ==========================================
best_model_path = os.path.join(SAVE_DIR, 'model_synth_real_cui_epoch_6.pt')

if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path, map_location=device))
    print("✅ Đã nạp thành công bộ não Đỉnh Cao (Epoch 6)!")
else:
    print("⚠️ Chưa tìm thấy file Epoch 6. Bạn kiểm tra lại tên file nhé!")
model.eval()

# ==========================================
# 2. HÀM SINH CÂU (BEAM SEARCH THUẦN TÚY)
# ==========================================
def generate_caption_optimized(model, tokenizer, pixel_values, max_length=60):
    with torch.no_grad():
        encoder_outputs = model.encoder(pixel_values=pixel_values)
        image_features = encoder_outputs.last_hidden_state[:, 0, :]
        projected_image = model.bridge(image_features).unsqueeze(1)
        
        generated_ids = model.decoder.generate(
            inputs_embeds=projected_image,
            max_new_tokens=max_length,
            num_beams=5,               # Quay lại 5 nhánh (Dò logic và kỹ hơn)
            no_repeat_ngram_size=2,    
            early_stopping=True,
            pad_token_id=tokenizer.eos_token_id
            # Đã gỡ bỏ do_sample và temperature để tối đa hóa điểm MedCAT/Align
        )
        caption = tokenizer.decode(generated_ids[0], skip_special_tokens=True).strip()
        return caption

# ==========================================
# 3. CHẠY VÀ ĐÓNG GÓI CHECKPOINT THÔNG MINH
# ==========================================
print("🚀 Đang quét các file Checkpoint Epoch 6 để gom tiến độ...")
processed_ids = set()
all_results = []

csv_files = glob.glob(os.path.join(SAVE_DIR, 'submission_epoch6_m*.csv'))

for file in csv_files:
    try:
        df_temp = pd.read_csv(file)
        for _, row in df_temp.iterrows():
            if row['ID'] not in processed_ids:
                processed_ids.add(row['ID'])
                all_results.append([row['ID'], row['Caption']])
    except Exception as e:
        pass

print(f"✅ Đã gom thành công {len(processed_ids)} câu từ các file Checkpoint!")

if len(all_results) > 0:
    temp_df = pd.DataFrame(all_results, columns=['ID', 'Caption'])
    temp_df.to_csv(MAIN_FILE, index=False, quoting=csv.QUOTE_ALL)

remaining_df = valid_df[~valid_df['ID'].isin(processed_ids)].reset_index(drop=True)

if len(remaining_df) == 0:
    print("🎉 Mọi thứ đã hoàn thiện 100%, không cần chạy thêm!")
else:
    print(f"🔥 Bắt đầu rặn {len(remaining_df)} câu bằng não Epoch 6...")
    total_done = len(processed_ids)
    
    with open(MAIN_FILE, mode='a', newline='', encoding='utf-8') as f:
        writer = csv.writer(f, quoting=csv.QUOTE_ALL)
        
        # Chỉ ghi Header nếu chạy lần đầu
        if total_done == 0:
            writer.writerow(['ID', 'Caption'])
            
        for idx in tqdm(range(len(remaining_df))):
            img_id = remaining_df.loc[idx, 'ID']
            img_path = os.path.join(IMG_DIR, f"{img_id}.jpg")
            
            try:
                image = Image.open(img_path).convert('RGB')
                pixel_values = basic_transform(image).unsqueeze(0).to(device) 
                pred_caption = generate_caption_optimized(model, tokenizer, pixel_values)
            except Exception as e:
                pred_caption = ""
                
            writer.writerow([img_id, pred_caption])
            f.flush() 
            
            all_results.append([img_id, pred_caption])
            total_done += 1
            
            if total_done % SAVE_STEP == 0:
                milestone_num = total_done // SAVE_STEP
                checkpoint_name = f'submission_epoch6_m{milestone_num}_({total_done}_cau).csv'
                checkpoint_path = os.path.join(SAVE_DIR, checkpoint_name)
                
                pd.DataFrame(all_results, columns=['ID', 'Caption']).to_csv(checkpoint_path, index=False, quoting=csv.QUOTE_ALL)
                print(f"\n💾 Đã lưu mốc {total_done} câu vào: {checkpoint_name}")

    # Tạo file nộp bài chung cuộc
    final_path = os.path.join(SAVE_DIR, f'submission_epoch6_FINAL_({total_done}_cau).csv')
    pd.DataFrame(all_results, columns=['ID', 'Caption']).to_csv(final_path, index=False, quoting=csv.QUOTE_ALL)
    print(f"\n🎉 XONG TRỌN VẸN 100%! File mang đi thi nằm tại: {final_path}")